# RQ1 — expert arm: one-class SVM vs. presence-vs-background (100% expert point)

**Purpose.** This notebook is the "100% expert" row of RQ1's
effort-reduction curve
(`docs/research_questions.md#rq1--foundation-models-vs-conventional-approaches`):
*A. dealbata* expert polygons only, no citizen-science presences.

**This is not a new method-selection pass.** Both methods are already
chosen on the citizen-science arm:

- **One-class SVM** (`nu=0.05`), chosen over Isolation Forest in
  [`one_class_method_selection.ipynb`](one_class_method_selection.ipynb).
- **Random Forest** (primary) and **Logistic Regression** (reported
  baseline) for presence-vs-background, chosen in
  [`presence_background_citizen_science.ipynb`](presence_background_citizen_science.ipynb).

This notebook applies the same configurations, unchanged, to the
expert arm, to check whether the citizen-science choice holds up on a
different, much more spatially clustered label source — the expert
polygons fall in only **18** ten-km blocks (measured below; the "15"
figure in `docs/research_questions.md` was for the 1 849 polygons
inside the Coimbra district only, whereas the pilot embedding extent
and this notebook's spatial join pick up a few more blocks at its
edge — flagged here, not yet reconciled in `research_questions.md`),
against **85** for the citizen-science arm. It reuses the same
background pool and the same two metrics (ROC-AUC, continuous Boyce)
so the two arms are directly comparable at their respective ends of
the curve. Closes part of `docs/todo.md`'s "Implementar os dois
classificadores da RQ1 para o braço de especialistas".


## 1. Background pool and pilot embeddings

Reuses, without rebuilding, the same files as the citizen-science
notebooks: the 2 000-point shared background pool and the RQ1 pilot's
TESSERA embeddings (citizen-science, expert and old COS-negative
points — see `data/README.md`).


In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from src.embeddings import block_id, load_pilot_embeddings

EMB_DIR = ROOT / "data" / "processed" / "embeddings"
PILOT_DIR = EMB_DIR / "tessera_v1.1_2023_rq1_pilot"
BG_FILE = EMB_DIR / "tessera_v1.1_2023_rq1_background" / "background_points.npz"

if not BG_FILE.exists():
    raise FileNotFoundError(
        f"{BG_FILE} is missing — run one_class_method_selection.ipynb first, "
        "it builds the shared background pool this notebook reuses."
    )

pilot_df, pilot_emb = load_pilot_embeddings(PILOT_DIR)
bg = np.load(BG_FILE)
bg_x, bg_y, bg_emb = bg["x"], bg["y"], bg["emb"]
print(f"{len(pilot_df):,} pilot embedding points (presences + old COS negatives)")
print(f"background points with valid embeddings: {len(bg_emb):,}")


## 2. Expert presence pixels

Unlike citizen science, there is no CSV with a precomputed `pixel`
column for the expert data — the expert reference is polygons
(`expert_polygons_clean.gpkg`, `classifica = 1` only since 2026-11-06,
see `data/README.md`), not points. The RQ1 pilot embeddings already
cover every TESSERA pixel inside these polygons within the Coimbra
district (`data/README.md`'s "TESSERA embeddings 2023 (RQ1 pilot
points)" entry), so presence pixels are recovered here with a spatial
join against `pilot_df`'s coordinates, instead of a pixel-id lookup.


In [ ]:
expert = gpd.read_file(ROOT / "data" / "interim" / "expert_reference" / "expert_polygons_clean.gpkg").to_crs(32629)

pilot_points = gpd.GeoDataFrame(pilot_df, geometry=gpd.points_from_xy(pilot_df["x"], pilot_df["y"]), crs=32629)
in_expert = pilot_points.within(expert.union_all()).values

expert_pixels = pilot_df[in_expert]
expert_emb_all = pilot_emb[in_expert]
valid = ~np.isnan(expert_emb_all).any(axis=1)
pres_x, pres_y, pres_emb = expert_pixels["x"].values[valid], expert_pixels["y"].values[valid], expert_emb_all[valid]

print(f"expert polygons (classifica = 1): {len(expert):,}")
print(f"pilot pixels inside an expert polygon: {in_expert.sum():,}")
print(f"presences with a non-NaN embedding: {len(pres_emb):,}")


## 3. Spatial block holdout

Same 10 km UTM blocks as the citizen-science notebooks, but the
expert presences fall in only ~18 blocks, not the ~85 the background
pool spans. Drawing test blocks from the full 85-block universe (as
the citizen-science notebooks do) would mean most of the 20%
held-out blocks contain no expert presences at all, starving most
seeds of test presences. **Adaptation (my own, this notebook, not yet
confirmed by the supervisor):** test blocks are drawn only from the
blocks that actually contain expert presences; background points are
still drawn from the full pool, restricted to whichever of those test
blocks they fall in, and training still uses every other block's
background (same as before) — only the *candidate pool for which
blocks can be held out* changes, not how background is split once a
block is chosen.


In [ ]:
pres_block = block_id(pres_x, pres_y)
bg_block = block_id(bg_x, bg_y)
pres_blocks_unique = sorted(set(pres_block))
print(f"{len(pres_blocks_unique)} spatial blocks (10 km) containing expert presences")
print(f"{len(set(bg_block))} spatial blocks (10 km) containing background points")


## 4. One-class SVM (`nu=0.05`, already chosen)

Trained only on expert presence embeddings (no background), evaluated
against held-out expert presences and held-out background, same
metrics as `one_class_method_selection.ipynb`. The hyperparameter is
not re-swept here — this checks whether the citizen-science choice
transfers, not a fresh selection.


In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

from src.evaluation import continuous_boyce

oc_auc, oc_boyce = [], []
for seed in range(1, 6):
    rng = np.random.default_rng(seed)
    test_blocks = set(rng.choice(pres_blocks_unique, size=max(1, round(0.2 * len(pres_blocks_unique))), replace=False))
    pres_train = pres_emb[~np.isin(pres_block, list(test_blocks))]
    pres_test = pres_emb[np.isin(pres_block, list(test_blocks))]
    bg_test = bg_emb[np.isin(bg_block, list(test_blocks))]
    if len(pres_test) < 5 or len(bg_test) < 5:
        continue

    scaler = StandardScaler().fit(pres_train)
    pres_train_s, pres_test_s, bg_test_s = (scaler.transform(a) for a in (pres_train, pres_test, bg_test))

    model = OneClassSVM(kernel="rbf", nu=0.05, gamma="scale").fit(pres_train_s)
    score_pres = model.decision_function(pres_test_s)
    score_bg = model.decision_function(bg_test_s)
    y_true = np.concatenate([np.ones(len(score_pres)), np.zeros(len(score_bg))])
    y_score = np.concatenate([score_pres, score_bg])
    oc_auc.append(roc_auc_score(y_true, y_score))
    oc_boyce.append(continuous_boyce(score_pres, score_bg))

print(f"n_splits: {len(oc_auc)}")
print(f"OneClassSVM ROC-AUC: {np.mean(oc_auc):.3f} (sd {np.std(oc_auc):.3f})")
print(f"OneClassSVM Boyce:   {np.mean(oc_boyce):.3f} (sd {np.std(oc_boyce):.3f})")


## 5. Presence-vs-background: Random Forest vs. Logistic Regression

Same configurations as `presence_background_citizen_science.ipynb`
(`class_weight="balanced"`, `n_estimators=300` for the forest,
`max_iter=1000` for Logistic Regression), same metrics, same adapted
spatial-block holdout as above.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

pb_configs = [
    ("RandomForest", lambda: RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)),
    ("LogisticRegression", lambda: LogisticRegression(class_weight="balanced", max_iter=1000)),
]

pb_results = {name: {"auc": [], "boyce": []} for name, _ in pb_configs}
for seed in range(1, 6):
    rng = np.random.default_rng(seed)
    test_blocks = set(rng.choice(pres_blocks_unique, size=max(1, round(0.2 * len(pres_blocks_unique))), replace=False))
    train_mask_pres, test_mask_pres = ~np.isin(pres_block, list(test_blocks)), np.isin(pres_block, list(test_blocks))
    train_mask_bg, test_mask_bg = ~np.isin(bg_block, list(test_blocks)), np.isin(bg_block, list(test_blocks))

    pres_train, pres_test = pres_emb[train_mask_pres], pres_emb[test_mask_pres]
    bg_train, bg_test = bg_emb[train_mask_bg], bg_emb[test_mask_bg]
    if len(pres_test) < 5 or len(bg_test) < 5:
        continue

    X_train = np.concatenate([pres_train, bg_train])
    y_train = np.concatenate([np.ones(len(pres_train)), np.zeros(len(bg_train))])
    X_test = np.concatenate([pres_test, bg_test])
    y_test = np.concatenate([np.ones(len(pres_test)), np.zeros(len(bg_test))])

    scaler = StandardScaler().fit(X_train)
    X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

    for name, ctor in pb_configs:
        model = ctor().fit(X_train_s, y_train)
        score = model.predict_proba(X_test_s)[:, 1]
        score_pres, score_bg = score[y_test == 1], score[y_test == 0]
        pb_results[name]["auc"].append(roc_auc_score(y_test, score))
        pb_results[name]["boyce"].append(continuous_boyce(score_pres, score_bg))

summary = pd.DataFrame(
    {
        "ROC-AUC (mean)": [np.mean(pb_results[n]["auc"]) for n, _ in pb_configs],
        "ROC-AUC (sd)": [np.std(pb_results[n]["auc"]) for n, _ in pb_configs],
        "Boyce (mean)": [np.mean(pb_results[n]["boyce"]) for n, _ in pb_configs],
        "Boyce (sd)": [np.std(pb_results[n]["boyce"]) for n, _ in pb_configs],
        "n_splits": [len(pb_results[n]["auc"]) for n, _ in pb_configs],
    },
    index=[n for n, _ in pb_configs],
)
summary.round(3)


## 6. Comparison and decision

**Result (expert arm, 100% expert point):** One-Class SVM ROC-AUC
0.911 (sd 0.037), Boyce 0.868 (sd 0.080); Random Forest ROC-AUC 0.990
(sd 0.004), Boyce 0.954 (sd 0.011); Logistic Regression ROC-AUC 0.990
(sd 0.003), Boyce 0.879 (sd 0.086) — all over 5 spatial-block splits,
all 5 valid (no split skipped for too few test points).

**Both citizen-science choices hold up:** presence-vs-background still
beats the one-class SVM on both metrics, and Random Forest still edges
out Logistic Regression on Boyce (0.954 vs. 0.879) with much lower
variance (sd 0.011 vs. 0.086), while the two are statistically tied on
ROC-AUC (0.990 vs. 0.990) — the same pattern as the citizen-science
arm, now with a clearer gap on Boyce. **Random Forest stays the
primary presence-vs-background classifier for the rest of RQ1**
(still my own choice, not confirmed by the supervisor); Logistic
Regression stays the reported baseline.

**Everything scores much higher here than on the citizen-science arm**
(e.g. one-class SVM: 0.911/0.868 here vs. 0.654/0.816 there; Random
Forest: 0.990/0.954 here vs. 0.842/0.989 there — Boyce is close in
both, but ROC-AUC jumps a lot). **Reading this as "the expert arm is
simply easier for the model" would be premature (my own interpretation,
not confirmed):** expert presences are large, contiguous forest
polygons with internally similar TESSERA embeddings, while the
background pool is uniform random points spread over mixed land
cover — separating "inside a big homogeneous acacia patch" from "a
random point anywhere in the district" may be an easier geometric
problem in embedding space than separating noisy, pixel-sized
citizen-science observations from the same background, independent of
how well either model actually captures acacia presence ecologically.
This is a hypothesis to keep in mind once the effort-reduction curve
mixes the two sources, not a conclusion.

**Caveats (first pass, not a definitive benchmark):**

- Expert *A. dealbata* polygons only — the mixed curve levels are not
  tested here.
- 18 spatial blocks, not 85 — the "15" figure in
  `docs/research_questions.md` referred to the Coimbra-district-only
  subset; this run's spatial join (pilot embeddings ∩ all 2 470
  `classifica = 1` polygons) reaches a few more blocks at the pilot
  extent's edge (not yet reconciled in that doc).
- Test-block sampling is restricted to presence-bearing blocks (see
  Section 3) — an adaptation needed because of the block scarcity
  above, not used on the citizen-science arm, and not yet confirmed
  by the supervisor.
- The background pool (uniform random, 50 m buffer) is the same one
  used throughout RQ1, still not confirmed by the supervisor as the
  final shared design.
- Hyperparameters are not tuned — same defaults as the citizen-science
  notebooks, no re-sweep for this arm.
- The compact neural network candidate from `docs/research_questions.md`
  is not tested here.
